# B2-022-probabilistic-latent-models — Practice p04 — Solution

**Type:** mc · **Difficulty:** intro · **Concepts:** autoencoder

*20 minutes.*  
**Set:** A  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Solution

Shape ledger for a batch of $B=32$:

| stage | module | output shape |
|---|---|---|
| input `x` | — | `(32, 20)` |
| encoder | `Linear(20, 8)` + ReLU | `(32, 8)` |
| **bottleneck (code)** | `Linear(8, 3)` | **`(32, 3)`** |
| decoder | `Linear(3, 8)` + ReLU | `(32, 8)` |
| reconstruction `x_hat` | `Linear(8, 20)` | `(32, 20)` |

A `Linear` layer acts on the last axis only, so the batch axis 32 is preserved at every stage.
An autoencoder is trained to reproduce its input, so the reconstruction target is `x` itself, shape `(32, 20)` (no labels are involved).
The code width $3<20$, so this autoencoder is **undercomplete**.

**Answer: B.**

In [ ]:
import torch
from torch import nn

torch.manual_seed(20260927)
encoder = nn.Sequential(nn.Linear(20, 8), nn.ReLU(), nn.Linear(8, 3))
decoder = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 20))
x = torch.randn(32, 20)
ledger = [("input", tuple(x.shape))]
h = x
for name, module in zip(("enc_linear", "enc_relu", "code"), encoder):
    h = module(h)
    ledger.append((name, tuple(h.shape)))
code = h
for name, module in zip(("dec_linear", "dec_relu", "reconstruction"), decoder):
    h = module(h)
    ledger.append((name, tuple(h.shape)))
x_hat = h
loss = nn.functional.mse_loss(x_hat, x)   # target is x itself
ANSWER = "B"
print(ledger)

### Answer check

In [ ]:
assert tuple(code.shape) == (32, 3)
assert x_hat.shape == x.shape == (32, 20)
assert x.dtype == torch.float32 and x_hat.dtype == torch.float32
assert code.shape[1] < x.shape[1]       # undercomplete
assert loss.ndim == 0
assert ANSWER == "B"